In [1]:
%pip install mysql-connector-python 


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import tkinter as tk
from tkinter import messagebox, ttk
import mysql.connector


# =========================================================================
#                          LOGIN & AUTH SYSTEM
# =========================================================================
class LoginWindow:

    def __init__(self, root):
        self.root = root
        self.root.title("Pharmacy System - Authentication")
        self.root.geometry("400x480")
        self.root.configure(bg="#f4f6f9")
        self.root.resizable(False, False)

        # Variables
        self.username = tk.StringVar()
        self.password = tk.StringVar()

        # Database Setup
        self.conn = None
        self.cursor = None
        self.init_db()

        # Build UI
        self.create_widgets()

    def init_db(self):
        """Connect to MySQL and create users table with default admin."""
        ports = [3306, 3308]
        for port in ports:
            try:
                self.conn = mysql.connector.connect(
                    host="localhost", user="root", password="", port=port
                )
                self.cursor = self.conn.cursor()

                self.cursor.execute("CREATE DATABASE IF NOT EXISTS pharmacy_db")
                self.conn.database = "pharmacy_db"

                # Users Table
                self.cursor.execute(
                    """
                    CREATE TABLE IF NOT EXISTS users (
                        id INT AUTO_INCREMENT PRIMARY KEY,
                        username VARCHAR(100) UNIQUE NOT NULL,
                        password VARCHAR(100) NOT NULL
                    )
                """
                )

                # Default Admin Account
                self.cursor.execute(
                    "INSERT IGNORE INTO users (username, password) VALUES ('admin', 'admin123')"
                )
                self.conn.commit()
                return
            except mysql.connector.Error:
                continue

        messagebox.showerror(
            "Database Error",
            "Could not connect to MySQL server via WAMP.\nEnsure WAMP Server is running.",
        )

    def create_widgets(self):
        # Top Header Banner
        header_frame = tk.Frame(self.root, bg="#0f2537", height=70)
        header_frame.pack(fill=tk.X, side=tk.TOP)

        header_label = tk.Label(
            header_frame,
            text="⚡ System Access",
            font=("Helvetica", 16, "bold"),
            fg="#00a8ff",
            bg="#0f2537",
        )
        header_label.pack(pady=20)

        # Form Container
        form_frame = tk.Frame(self.root, bg="#f4f6f9")
        form_frame.pack(pady=20, padx=30, fill=tk.BOTH, expand=True)

        # Username
        tk.Label(
            form_frame,
            text="Username:",
            font=("Helvetica", 10, "bold"),
            bg="#f4f6f9",
            fg="#2c3e50",
            anchor="w",
        ).pack(fill=tk.X, pady=(10, 2))

        entry_user = tk.Entry(
            form_frame,
            textvariable=self.username,
            font=("Helvetica", 11),
            bd=1,
            relief=tk.SOLID,
        )
        entry_user.pack(fill=tk.X, ipady=5, pady=(0, 15))

        # Password
        tk.Label(
            form_frame,
            text="Password:",
            font=("Helvetica", 10, "bold"),
            bg="#f4f6f9",
            fg="#2c3e50",
            anchor="w",
        ).pack(fill=tk.X, pady=(0, 2))

        entry_pass = tk.Entry(
            form_frame,
            textvariable=self.password,
            show="*",
            font=("Helvetica", 11),
            bd=1,
            relief=tk.SOLID,
        )
        entry_pass.pack(fill=tk.X, ipady=5, pady=(0, 20))
        entry_pass.bind("<Return>", lambda event: self.login())

        # Buttons
        btn_login = tk.Button(
            form_frame,
            text="Login",
            font=("Helvetica", 11, "bold"),
            bg="#27ae60",
            fg="white",
            bd=0,
            cursor="hand2",
            command=self.login,
        )
        btn_login.pack(fill=tk.X, ipady=8, pady=(0, 10))

        btn_register = tk.Button(
            form_frame,
            text="Register New Account",
            font=("Helvetica", 11, "bold"),
            bg="#2980b9",
            fg="white",
            bd=0,
            cursor="hand2",
            command=self.open_register_window,
        )
        btn_register.pack(fill=tk.X, ipady=8)

        # Default Admin Hint Label
        tk.Label(
            form_frame,
            text="Default Admin: admin / admin123",
            font=("Helvetica", 9, "italic"),
            bg="#f4f6f9",
            fg="#7f8c8d",
        ).pack(pady=(20, 0))

    def login(self):
        user = self.username.get().strip()
        pwd = self.password.get().strip()

        if not user or not pwd:
            messagebox.showwarning(
                "Input Error", "Please enter both Username and Password!"
            )
            return

        if not self.cursor:
            messagebox.showerror(
                "Database Error", "Not connected to database."
            )
            return

        try:
            self.cursor.execute(
                "SELECT * FROM users WHERE username=%s AND password=%s",
                (user, pwd),
            )
            result = self.cursor.fetchone()

            if result:
                messagebox.showinfo("Access Granted", f"Welcome back, {user}!")
                self.close_db()
                self.root.destroy()  # Close login window

                # Launch Main Application
                main_root = tk.Tk()
                PharmacyManagementSystem(main_root)
                main_root.mainloop()
            else:
                messagebox.showerror(
                    "Access Denied", "Invalid Username or Password!"
                )
        except mysql.connector.Error as e:
            messagebox.showerror("Error", f"Login failed: {e}")

    def open_register_window(self):
        RegWindow(self.root, self.cursor, self.conn)

    def close_db(self):
        if self.cursor:
            self.cursor.close()
        if self.conn and self.conn.is_connected():
            self.conn.close()


# =========================================================================
#                        USER REGISTRATION POPUP
# =========================================================================
class RegWindow:

    def __init__(self, parent, cursor, conn):
        self.top = tk.Toplevel(parent)
        self.top.title("Register New User")
        self.top.geometry("350x320")
        self.top.configure(bg="#ffffff")
        self.top.grab_set()  # Make window modal

        self.cursor = cursor
        self.conn = conn

        self.new_user = tk.StringVar()
        self.new_pass = tk.StringVar()
        self.confirm_pass = tk.StringVar()

        self.create_widgets()

    def create_widgets(self):
        tk.Label(
            self.top,
            text="Create Account",
            font=("Helvetica", 14, "bold"),
            bg="#ffffff",
            fg="#2c3e50",
        ).pack(pady=15)

        form = tk.Frame(self.top, bg="#ffffff")
        form.pack(padx=20, fill=tk.BOTH, expand=True)

        tk.Label(
            form, text="New Username:", font=("Helvetica", 9, "bold"), bg="white"
        ).pack(anchor="w")
        tk.Entry(
            form,
            textvariable=self.new_user,
            font=("Helvetica", 10),
            bd=1,
            relief=tk.SOLID,
        ).pack(fill=tk.X, ipady=3, pady=(0, 10))

        tk.Label(
            form, text="New Password:", font=("Helvetica", 9, "bold"), bg="white"
        ).pack(anchor="w")
        tk.Entry(
            form,
            textvariable=self.new_pass,
            show="*",
            font=("Helvetica", 10),
            bd=1,
            relief=tk.SOLID,
        ).pack(fill=tk.X, ipady=3, pady=(0, 10))

        tk.Label(
            form,
            text="Confirm Password:",
            font=("Helvetica", 9, "bold"),
            bg="white",
        ).pack(anchor="w")
        tk.Entry(
            form,
            textvariable=self.confirm_pass,
            show="*",
            font=("Helvetica", 10),
            bd=1,
            relief=tk.SOLID,
        ).pack(fill=tk.X, ipady=3, pady=(0, 15))

        tk.Button(
            form,
            text="Submit Registration",
            font=("Helvetica", 10, "bold"),
            bg="#27ae60",
            fg="white",
            bd=0,
            cursor="hand2",
            command=self.register,
        ).pack(fill=tk.X, ipady=6)

    def register(self):
        u = self.new_user.get().strip()
        p1 = self.new_pass.get().strip()
        p2 = self.confirm_pass.get().strip()

        if not u or not p1 or not p2:
            messagebox.showwarning("Warning", "All fields are required!")
            return

        if p1 != p2:
            messagebox.showwarning("Warning", "Passwords do not match!")
            return

        try:
            self.cursor.execute(
                "INSERT INTO users (username, password) VALUES (%s, %s)",
                (u, p1),
            )
            self.conn.commit()
            messagebox.showinfo(
                "Success", "Account created successfully! You can now log in."
            )
            self.top.destroy()
        except mysql.connector.Error as e:
            if e.errno == 1062:  # Duplicate entry
                messagebox.showerror(
                    "Error", "Username already exists! Choose another."
                )
            else:
                messagebox.showerror("Error", f"Failed to register: {e}")


# =========================================================================
#                     MAIN PHARMACY SYSTEM DASHBOARD
# =========================================================================
class PharmacyManagementSystem:

    def __init__(self, root):
        self.root = root
        self.root.title("Pharmacy Management System")
        self.root.geometry("1100x650")
        self.root.configure(bg="#f4f6f9")

        # Variables
        self.med_id = tk.StringVar()
        self.med_name = tk.StringVar()
        self.company = tk.StringVar()
        self.price = tk.StringVar()
        self.qty = tk.StringVar()
        self.exp_date = tk.StringVar()
        self.search_term = tk.StringVar()

        self.conn = None
        self.cursor = None

        # Clean shutdown protocol
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

        # Build UI Components
        self.create_header()
        self.create_form_frame()
        self.create_table_frame()

        # Database Setup & Initial Fetch
        if self.init_db():
            self.fetch_data()

    def init_db(self):
        """Initialize MySQL database connection for inventory."""
        ports = [3306, 3308]
        for port in ports:
            try:
                self.conn = mysql.connector.connect(
                    host="localhost", user="root", password="", port=port
                )
                self.cursor = self.conn.cursor()

                self.cursor.execute("CREATE DATABASE IF NOT EXISTS pharmacy_db")
                self.conn.database = "pharmacy_db"

                self.cursor.execute(
                    """
                    CREATE TABLE IF NOT EXISTS inventory (
                        id INT AUTO_INCREMENT PRIMARY KEY,
                        name VARCHAR(150) NOT NULL,
                        company VARCHAR(150) NOT NULL,
                        price DECIMAL(10, 2) NOT NULL CHECK (price > 0),
                        qty INT NOT NULL CHECK (qty >= 0),
                        exp_date VARCHAR(10) NOT NULL
                    )
                """
                )
                self.conn.commit()
                return True
            except mysql.connector.Error:
                continue

        messagebox.showerror(
            "Database Error", "Could not connect to MySQL server via WAMP."
        )
        return False

    def create_header(self):
        header_frame = tk.Frame(self.root, bg="#2c3e50", height=60)
        header_frame.pack(fill=tk.X, side=tk.TOP)

        title_label = tk.Label(
            header_frame,
            text="Pharmacy Management System",
            font=("Helvetica", 18, "bold"),
            fg="white",
            bg="#2c3e50",
        )
        title_label.pack(pady=15)

    def create_form_frame(self):
        form_frame = tk.LabelFrame(
            self.root,
            text=" Medicine Details ",
            font=("Helvetica", 11, "bold"),
            bg="#ffffff",
            fg="#2c3e50",
            bd=2,
            relief=tk.GROOVE,
        )
        form_frame.place(x=20, y=80, width=380, height=540)

        fields = [
            ("Medicine Name:", self.med_name, 0),
            ("Company / Brand:", self.company, 1),
            ("Price (₹):", self.price, 2),
            ("Quantity:", self.qty, 3),
            ("Expiry Date (MM/YY):", self.exp_date, 4),
        ]

        for label_text, var, row in fields:
            lbl = tk.Label(
                form_frame,
                text=label_text,
                font=("Helvetica", 10, "bold"),
                bg="white",
                anchor="w",
            )
            lbl.grid(
                row=row * 2,
                column=0,
                columnspan=2,
                sticky="w",
                padx=15,
                pady=(10, 2),
            )

            entry = tk.Entry(
                form_frame,
                textvariable=var,
                font=("Helvetica", 10),
                bd=1,
                relief=tk.SOLID,
            )
            entry.grid(
                row=row * 2 + 1,
                column=0,
                columnspan=2,
                sticky="ew",
                padx=15,
                ipady=4,
            )

        form_frame.columnconfigure(0, weight=1)

        btn_frame = tk.Frame(form_frame, bg="white")
        btn_frame.place(x=15, y=360, width=348, height=150)

        btn_style = {
            "font": ("Helvetica", 10, "bold"),
            "fg": "white",
            "bd": 0,
            "cursor": "hand2",
        }

        tk.Button(
            btn_frame,
            text="Add Medicine",
            bg="#27ae60",
            command=self.add_medicine,
            **btn_style,
        ).grid(row=0, column=0, padx=5, pady=5, sticky="ew")

        tk.Button(
            btn_frame,
            text="Update Selected",
            bg="#2980b9",
            command=self.update_medicine,
            **btn_style,
        ).grid(row=0, column=1, padx=5, pady=5, sticky="ew")

        tk.Button(
            btn_frame,
            text="Delete Selected",
            bg="#c0392b",
            command=self.delete_medicine,
            **btn_style,
        ).grid(row=1, column=0, padx=5, pady=5, sticky="ew")

        tk.Button(
            btn_frame,
            text="Clear Form",
            bg="#7f8c8d",
            command=self.clear_fields,
            **btn_style,
        ).grid(row=1, column=1, padx=5, pady=5, sticky="ew")

        btn_frame.columnconfigure(0, weight=1)
        btn_frame.columnconfigure(1, weight=1)

    def create_table_frame(self):
        table_frame = tk.Frame(self.root, bg="#ffffff", bd=2, relief=tk.GROOVE)
        table_frame.place(x=420, y=80, width=650, height=540)

        search_bar = tk.Frame(table_frame, bg="#ffffff")
        search_bar.pack(fill=tk.X, padx=10, pady=10)

        tk.Label(
            search_bar,
            text="Search Name:",
            font=("Helvetica", 10, "bold"),
            bg="white",
        ).pack(side=tk.LEFT, padx=5)

        search_entry = tk.Entry(
            search_bar,
            textvariable=self.search_term,
            font=("Helvetica", 10),
            bd=1,
            relief=tk.SOLID,
        )
        search_entry.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=5, ipady=2)
        search_entry.bind("<Return>", lambda event: self.search_medicine())

        tk.Button(
            search_bar,
            text="Search",
            bg="#34495e",
            fg="white",
            font=("Helvetica", 9, "bold"),
            command=self.search_medicine,
            bd=0,
            padx=10,
        ).pack(side=tk.LEFT, padx=2)

        tk.Button(
            search_bar,
            text="Reset",
            bg="#95a5a6",
            fg="white",
            font=("Helvetica", 9, "bold"),
            command=self.fetch_data,
            bd=0,
            padx=10,
        ).pack(side=tk.LEFT, padx=2)

        scroll_y = tk.Scrollbar(table_frame, orient=tk.VERTICAL)
        scroll_x = tk.Scrollbar(table_frame, orient=tk.HORIZONTAL)

        columns = ("ID", "Name", "Company", "Price", "Qty", "Exp Date")
        self.tree = ttk.Treeview(
            table_frame,
            columns=columns,
            show="headings",
            yscrollcommand=scroll_y.set,
            xscrollcommand=scroll_x.set,
        )

        scroll_y.pack(side=tk.RIGHT, fill=tk.Y)
        scroll_x.pack(side=tk.BOTTOM, fill=tk.X)

        scroll_y.config(command=self.tree.yview)
        scroll_x.config(command=self.tree.xview)

        self.tree.heading("ID", text="ID")
        self.tree.heading("Name", text="Medicine Name")
        self.tree.heading("Company", text="Company")
        self.tree.heading("Price", text="Price (₹)")
        self.tree.heading("Qty", text="Stock Qty")
        self.tree.heading("Exp Date", text="Expiry")

        self.tree.column("ID", width=40, anchor="center")
        self.tree.column("Name", width=140)
        self.tree.column("Company", width=120)
        self.tree.column("Price", width=70, anchor="center")
        self.tree.column("Qty", width=70, anchor="center")
        self.tree.column("Exp Date", width=90, anchor="center")

        self.tree.pack(fill=tk.BOTH, expand=True, padx=10, pady=(0, 10))
        self.tree.bind("<ButtonRelease-1>", self.get_selected_row)

    # ------------------ MySQL CRUD Functions ------------------

    def add_medicine(self):
        if not self.cursor:
            messagebox.showerror(
                "Error", "Database is not connected. Start WAMP Server."
            )
            return

        if not self.validate_inputs():
            return

        try:
            sql = """
                INSERT INTO inventory (name, company, price, qty, exp_date) 
                VALUES (%s, %s, %s, %s, %s)
            """
            values = (
                self.med_name.get().strip(),
                self.company.get().strip(),
                float(self.price.get()),
                int(self.qty.get()),
                self.exp_date.get().strip(),
            )
            self.cursor.execute(sql, values)
            self.conn.commit()

            messagebox.showinfo("Success", "Medicine added successfully!")
            self.fetch_data()
            self.clear_fields()
        except mysql.connector.Error as e:
            messagebox.showerror("Database Error", f"Failed to add item: {e}")

    def fetch_data(self):
        if not self.cursor:
            return

        try:
            self.cursor.execute("SELECT * FROM inventory")
            rows = self.cursor.fetchall()

            self.tree.delete(*self.tree.get_children())
            for row in rows:
                self.tree.insert("", tk.END, values=row)

            self.search_term.set("")
        except mysql.connector.Error as e:
            messagebox.showerror("Database Error", f"Failed to fetch data: {e}")

    def get_selected_row(self, event):
        selected_item = self.tree.focus()
        if not selected_item:
            return

        content = self.tree.item(selected_item)
        row = content["values"]

        if row:
            self.med_id.set(row[0])
            self.med_name.set(row[1])
            self.company.set(row[2])
            self.price.set(row[3])
            self.qty.set(row[4])
            self.exp_date.set(row[5])

    def update_medicine(self):
        if not self.cursor:
            return

        if not self.med_id.get():
            messagebox.showwarning(
                "Warning", "Please select a record from the table to update."
            )
            return

        if not self.validate_inputs():
            return

        try:
            sql = """
                UPDATE inventory 
                SET name=%s, company=%s, price=%s, qty=%s, exp_date=%s
                WHERE id=%s
            """
            values = (
                self.med_name.get().strip(),
                self.company.get().strip(),
                float(self.price.get()),
                int(self.qty.get()),
                self.exp_date.get().strip(),
                int(self.med_id.get()),
            )
            self.cursor.execute(sql, values)
            self.conn.commit()

            messagebox.showinfo("Success", "Medicine updated successfully!")
            self.fetch_data()
            self.clear_fields()
        except mysql.connector.Error as e:
            messagebox.showerror("Database Error", f"Failed to update: {e}")

    def delete_medicine(self):
        if not self.cursor:
            return

        if not self.med_id.get():
            messagebox.showwarning(
                "Warning", "Please select a record to delete."
            )
            return

        confirm = messagebox.askyesno(
            "Confirm Delete", "Are you sure you want to delete this record?"
        )
        if confirm:
            try:
                sql = "DELETE FROM inventory WHERE id=%s"
                self.cursor.execute(sql, (int(self.med_id.get()),))
                self.conn.commit()

                messagebox.showinfo(
                    "Success", "Medicine deleted successfully!"
                )
                self.fetch_data()
                self.clear_fields()
            except mysql.connector.Error as e:
                messagebox.showerror("Database Error", f"Failed to delete: {e}")

    def search_medicine(self):
        if not self.cursor:
            return

        query = self.search_term.get().strip()
        if not query:
            self.fetch_data()
            return

        try:
            sql = "SELECT * FROM inventory WHERE name LIKE %s OR company LIKE %s"
            wildcard = f"%{query}%"
            self.cursor.execute(sql, (wildcard, wildcard))
            rows = self.cursor.fetchall()

            self.tree.delete(*self.tree.get_children())
            for row in rows:
                self.tree.insert("", tk.END, values=row)
        except mysql.connector.Error as e:
            messagebox.showerror("Database Error", f"Failed to search: {e}")

    def clear_fields(self):
        self.med_id.set("")
        self.med_name.set("")
        self.company.set("")
        self.price.set("")
        self.qty.set("")
        self.exp_date.set("")

        selected_items = self.tree.selection()
        if selected_items:
            self.tree.selection_remove(selected_items)

    def validate_inputs(self):
        if (
            not self.med_name.get().strip()
            or not self.company.get().strip()
            or not self.price.get().strip()
            or not self.qty.get().strip()
            or not self.exp_date.get().strip()
        ):
            messagebox.showwarning("Warning", "All fields are required!")
            return False

        try:
            p = float(self.price.get())
            if p <= 0:
                messagebox.showwarning(
                    "Warning", "Price must be greater than zero."
                )
                return False
        except ValueError:
            messagebox.showwarning("Warning", "Price must be a valid number.")
            return False

        try:
            q = int(self.qty.get())
            if q < 0:
                messagebox.showwarning(
                    "Warning", "Quantity cannot be negative."
                )
                return False
        except ValueError:
            messagebox.showwarning("Warning", "Quantity must be an integer.")
            return False

        exp_pattern = r"^(0[1-9]|1[0-2])\/(\d{2}|\d{4})$"
        if not re.match(exp_pattern, self.exp_date.get().strip()):
            messagebox.showwarning(
                "Warning", "Expiry Date must be in MM/YY or MM/YYYY format."
            )
            return False

        return True

    def on_closing(self):
        if hasattr(self, "cursor") and self.cursor:
            self.cursor.close()
        if hasattr(self, "conn") and self.conn and self.conn.is_connected():
            self.conn.close()
        self.root.destroy()


# =========================================================================
#                             APPLICATION LAUNCH
# =========================================================================
if __name__ == "__main__":
    login_root = tk.Tk()
    auth_app = LoginWindow(login_root)

    # Bring Login window to front
    login_root.lift()
    login_root.attributes("-topmost", True)
    login_root.after_idle(login_root.attributes, "-topmost", False)

    login_root.mainloop()